## Concurrencia

*Python ofrece módulos que proveen soporte para la ejecución concurrente de código. La elección de qué herramienta utilizar depende de la tarea a ejecutar (vinculada a CPU o vinculada a E/S) y del estilo preferido de desarrollo (multi-tarea cooperativa o multi-tarea apropiativa).*

En este notebook vamos a ver la ejecución concurrente de [hebras](https://docs.python.org/es/3/library/threading.html#thread-objects) mediante [cerrojos](https://docs.python.org/es/3/library/threading.html#lock-objects) que permitirán definir zonas de exclusión mutua. Además veremos cómo comunicar entre sí las hebras mediante el usi de [eventos](https://docs.python.org/es/3/library/threading.html#event-objects).

Los elementos que necesitamos importar (**Thread**, **Event** y **Lock**) están en el módulo [theading](https://docs.python.org/es/3/library/threading.html).

### Thread

Para lanzar en Python una hebra que sea ejecutada en segundo plano necesitamos definir su funcionalidad dentro de una función. Una vez creada esta función instanciamos un objeto de tipo *Thread* pasando como argumentos el nombre de la función y uns lista opcional de argumentos.

```python
from threading import Thread

def funcion(param1, param2, param3):
    # Cuerpo de la función que realiza la tarea definida para la hebra

nuevahebra = Thread(target=funcion, args=[arg1, arg2, arg3])
```

Una vez definida la hebra, para lanzarla hay que invocar el método [start()](https://docs.python.org/es/3/library/threading.html#threading.Thread.start). Esta hebra permanecerá "viva" hasta que termine la ejecución de la función que la definió. En todo momento podemos comprobar su estado mediante el empleo del método [is_alive()](https://docs.python.org/es/3/library/threading.html#threading.Thread.is_alive). También podemos bloquear la ejecución de una hebra1 hasta que termine la hebra2. Para ello usaremos el método [join()](https://docs.python.org/es/3/library/threading.html#threading.Thread.join). La llamada *otrahebra.join()* detendrá la ejecución de la hebra desde la que se hace la llamada hasta que *otrahebra* termine. 

Es importante considerar que las variables definidas en la función son locales, por lo que para pasar datos de una hebra a otra vamos a usar [variables globales](https://www.w3schools.com/python/python_variables_global.asp).

### Lock

La ejecución concurrente de varias hebras presenta un problema evidente: el acceso simultáneo de varias hebras a un recurso compartido. Si una hebra modifica un recurso compartido **durante** la lectura de dicho recurso por parte de otra hebra provocamos un problema ya que los datos leidos serán inconsistentes. Este problema se evita definiendo zonas en las que el acceso concurrente está prohibido. Estas son las [zonas de exclusión mutua](https://es.wikipedia.org/wiki/Exclusi%C3%B3n_mutua_(inform%C3%A1tica)). Una de los mecanismos que Python ofrece para contruir zonas de exclusión mutua son los cerrojos.

Podríamos ver un cerrojo como una tarjeta de seguridad que permite el paso a una zona segura (exclusión mutua) solamente a quien posea la tarjeta. Quien no tenga tarjeta debe esperar a que quien actualmente posea la tarjeta la deje libre.

El cerrojo es un elemento de tipo *Lock*.

```python
from threading import Lock

cerrojo = Lock()
```

Si dos hebras quieren acceder a un recurso compartido *comp* protegido dentro de una zona de exclusión mutua, ambas tienen que "solicitar" el acceso mediante una llamada al método [adquire()](https://docs.python.org/es/3/library/threading.html#threading.Lock.acquire) antes de acceder al recurso compartido y deben "liberar" el acceso al terminar mediante una llamada al método [release()](https://docs.python.org/es/3/library/threading.html#threading.Lock.release).

```python
def hebra1():
    global comp
    cerrojo.adquire()
    #--- Zona de exclusión mutua para el acceso a comp
    #....
    #--- Fin de la zona de exclusión mutua
    cerrojo.release()

def hebra2():
    global comp
    cerrojo.adquire()
    #--- Zona de exclusión mutua para el acceso a comp
    #....
    #--- Fin de la zona de exclusión mutua
    cerrojo.release()
```

La primera hebra que haga la llamada a *cerrojo.adquire()* pasará a la zona de exclusión mutua. La otra hebra quedará retenida en su llamada a *cerrojo.adquire()* hasta que la otra libere el cerrojo con *release()*.

### Event

La comunicación más simple entre hebras la realizamos mediante eventos. Permite establecer un punto en una hebra en el que la ejecución se detendrá hasta que desde otra hebra se active un evento.

Los eventos son elementos de tipo *Event*.

```python
from threading import Event

evento = Event()
```

Un evento nos ofrece métodos que permiten una comunicación sencilla:

- [**set()**](https://docs.python.org/es/3/library/threading.html#threading.Event.set): Método utilizado para activar un evento.
- [**clear()**](https://docs.python.org/es/3/library/threading.html#threading.Event.clear): Método usado para desactivar un evento.
- [**wait(timeout=None)**](https://docs.python.org/es/3/library/threading.html#threading.Event.wait): Método que espera hasta que se active un evento o hasta que pase un tiempo límite.
- **is_set()**: Método que devuelve True si el evento está actualmente activo.

### Ejemplo

In [1]:
from threading import Thread, Event, Lock

In [2]:
import time
c = "*"   # Variable global a la que queremos acceder en mutex
l = Lock() # Cerrojo para el acceso mutex a c
e = Event() # Evento para anunciar el paso de 30 segundos
salir = Event()  # Evento para indicar que se debe salir del programa

In [3]:
# Cada x segundos imprime el número de segundo por el que va y alterna c entre * y +. Termina cuando llega a 60 segundos
def funchebra1(x):
    global c
    global e
    print("Incio de hebra 1")
    
    inicio = time.time()
    contador = 0
    while (time.time()-inicio)<60:
        if contador == 30:
            e.set()
        if contador % x == 0:
            print(contador)
            l.acquire()
            if c == "*":
                c = "+"
            else:
                c = "*"
            l.release()
        while (time.time()-inicio) < contador + 1:
            time.sleep(0.1)
        contador += 1
    print("Fin de hebra 1")

In [4]:
# Cada segundo imprime el valor de c. Termina cuando se activa el evento salir
def funchebra2():
    global salir
    global c
    print("Incio de hebra 2")
    while not salir.is_set():
        time.sleep(1)
        l.acquire()
        print(c)
        l.release()
    print("Fin de hebra 2")

In [5]:
# Espera a que se produzca el evento e para imprimir una línea
def funchebra3():
    global e
    print("Incio de hebra 3")
    e.wait()
    print("--------------------------------------------")
    e.clear()
    salir.set()
    print("Fin de hebra 3")

In [6]:
# Definimos las hebras (no se inicia la ejecución hasta que posteriormente se haga una llamada a start)
hebra1 = Thread(target=funchebra1, args=[5]) # A la hebra1 le pasamos el argumento 5
hebra2 = Thread(target=funchebra2)
hebra3 = Thread(target=funchebra3)

In [7]:
# Iniciamos las 3 hebras (en total habrá 4; la hebra actual más las 3 que estamos invocando)
hebra1.start()
hebra2.start()
hebra3.start()
# Una vez iniciadas las 3 hebras el programa sigue su camino

# Esperamos a que se produzca el evento salir para terminar el programa
#while not salir.is_set():    
#    pass
salir.wait()

print("Esperando a que acabe la hebra 1")
hebra1.join()
print("Final del programa")

Incio de hebra 1
0
Incio de hebra 2
Incio de hebra 3
+
+
+
+
+
5
*
*
*
*
*
10
+
+
+
+
+
15
*
*
*
*
*
20
+
+
+
+
25
*
*
*
*
*
*
30
--------------------------------------------
Fin de hebra 3
Esperando a que acabe la hebra 1
+
Fin de hebra 2
35
40
45
50
55
Fin de hebra 1
Final del programa
